# Milestone 2 — Data Preparation: Car Price Prediction

Google Colab → Files (folder on the left) → **Upload to session storage** *car_price_dataset.csv*

[Kaggle Dataset](https://www.kaggle.com/datasets/deepcontractor/car-price-prediction-challenge)

---

### Notebook Overview

This notebook prepares the raw car listings dataset for model training. The pipeline follows this order:

1. Load data & global configuration
2. Raw data inspection
3. Data cleaning (type fixes, hidden nulls, imputation)
4. Outlier investigation
5. **Train / Test split** ← done *before* any feature engineering to prevent data leakage
6. Feature engineering (hand-crafted features)
7. Analysis of new features
8. Encoding categorical variables
9. Scaling
10. Feature selection
11. Export

## 1. Imports & Global Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder
from sklearn.inspection import permutation_importance
from sklearn.ensemble import GradientBoostingRegressor

# Reproducibility — every random operation in this notebook uses this seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Consistent plot appearance
sns.set_theme(style='ticks', palette='Set2')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 4)})

print('Setup complete. Seed:', RANDOM_SEED)

## 2. Load & Inspect Raw Data

In [ ]:
df_raw = pd.read_csv('car_price_dataset.csv')
print('Shape:', df_raw.shape)
print('\nColumn dtypes:')
print(df_raw.dtypes)
df_raw.head(10)

In [ ]:
# Unique value counts give us a quick sense of each column's nature
print('Unique values per column:')
print(df_raw.nunique().sort_values())
print('\nSample values for object columns:')
for col in df_raw.select_dtypes('object').columns:
    print(f'  {col}: {df_raw[col].unique()[:6]}')

**Findings from inspection:**
- Several numeric columns are stored as strings (e.g. `Mileage` = "15000 km", `Engine volume` = "2.0 Turbo")
- The `Levy` column uses `"-"` as a placeholder for missing tax values instead of `NaN`
- `Price` also contains non-numeric entries that must be coerced
- `Doors` contains textual values like `"04-May"` due to spreadsheet date-parsing errors

## 3. Data Cleaning

All cleaning steps are applied to a working copy of the raw data to keep the original intact.

In [ ]:
df = df_raw.copy()

# --- Step 1: Replace dash placeholders with proper NaN ---
# The Levy column encodes missing tax as "-" instead of NaN.
# We standardise all such placeholders before any type conversion.
df.replace('-', np.nan, inplace=True)

# --- Step 2: Strip units and cast Mileage to integer ---
# Raw value example: "186005 km" → 186005
df['Mileage'] = (
    df['Mileage'].astype(str)
    .str.replace(' km', '', regex=False)
    .str.replace(',', '', regex=False)
)
df['Mileage'] = pd.to_numeric(df['Mileage'], errors='coerce')

# --- Step 3: Extract numeric engine displacement (handle "Turbo" suffix) ---
# Raw value example: "2.0 Turbo" → 2.0
# We store the turbo flag separately later as a hand-crafted feature.
df['Engine volume'] = (
    df['Engine volume'].astype(str)
    .str.extract(r'(\d+\.?\d*)')[0]
)
df['Engine volume'] = pd.to_numeric(df['Engine volume'], errors='coerce')

# --- Step 4: Clean Levy (remove thousands commas, cast to float) ---
df['Levy'] = (
    df['Levy'].astype(str)
    .str.replace(',', '', regex=False)
    .replace('nan', np.nan)
)
df['Levy'] = pd.to_numeric(df['Levy'], errors='coerce')

# --- Step 5: Clean Price ---
df['Price'] = (
    df['Price'].astype(str)
    .str.replace(',', '', regex=False)
)
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

# --- Step 6: Fix Doors column (Excel date-parse artifact) ---
# Some values appear as "04-May" instead of "4". We extract the leading number.
df['Doors'] = df['Doors'].astype(str).str.extract(r'(\d+)')[0]
df['Doors'] = pd.to_numeric(df['Doors'], errors='coerce')

# --- Step 7: Drop rows where the target (Price) is missing ---
before = len(df)
df.dropna(subset=['Price'], inplace=True)
print(f'Rows dropped — missing Price: {before - len(df)}')

print(f'\nShape after type cleaning: {df.shape}')
df.dtypes

### 3.1 Missing Value Analysis & Imputation

In [ ]:
# Count missing values after standardisation
missing = df.isnull().mean() * 100
missing = missing[missing > 0].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 3.5))
bars = ax.bar(missing.index, missing.values, color=sns.color_palette('Set2'))
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
ax.set_ylabel('Missing (%)')
ax.set_title('Missing Values by Column (after dash-replacement)')
ax.set_ylim(0, missing.max() * 1.25)
sns.despine()
plt.tight_layout()
plt.savefig('fig01_missing_values.png', bbox_inches='tight')
plt.show()

In [ ]:
# Impute Levy with its median value.
# Median is preferred over mean here because Levy is right-skewed —
# the mean is pulled upward by high-tax luxury vehicles, making it
# a poor representative of the typical missing entry.
levy_median = df['Levy'].median()
df['Levy'].fillna(levy_median, inplace=True)
print(f'Levy: {df["Levy"].isnull().sum()} missing after imputation (median = {levy_median:.0f})')

# For remaining numeric NaNs (Mileage, Engine volume, Doors), use column median
num_cols = df.select_dtypes(include=np.number).columns
for col in num_cols:
    n_miss = df[col].isnull().sum()
    if n_miss > 0:
        med = df[col].median()
        df[col].fillna(med, inplace=True)
        print(f'{col}: {n_miss} values imputed with median={med:.2f}')

# Drop rows where any categorical column is still missing
cat_cols = df.select_dtypes('object').columns
before = len(df)
df.dropna(subset=cat_cols, inplace=True)
print(f'\nRows dropped — missing categoricals: {before - len(df)}')
print(f'Final clean shape: {df.shape}')

## 4. Outlier Investigation

Rather than applying a blanket rule, we investigate each end of the price distribution separately.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: full distribution
axes[0].hist(df['Price'], bins=100, color=sns.color_palette('Set2')[0], edgecolor='none')
axes[0].set_title('Full Price Distribution (raw)')
axes[0].set_xlabel('Price ($)')
axes[0].set_ylabel('Count')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Right: log scale to see both ends clearly
axes[1].hist(np.log1p(df['Price']), bins=80, color=sns.color_palette('Set2')[1], edgecolor='none')
axes[1].set_title('log(1 + Price) Distribution')
axes[1].set_xlabel('log(1 + Price)')
axes[1].set_ylabel('Count')

plt.suptitle('Price Distribution — Before Outlier Filtering', fontweight='bold')
plt.tight_layout()
plt.savefig('fig02_price_distribution_raw.png', bbox_inches='tight')
plt.show()

print('Lowest 10 prices:')
print(df['Price'].nsmallest(10).values)
print('\nHighest 10 prices:')
print(df['Price'].nlargest(10).values)

In [ ]:
# --- Lower bound: remove prices below $500 ---
# Prices like $1, $2, $10 are clearly data entry errors with no real-world
# meaning. No functional used car is sold for under $500.

# --- Upper bound: we DO NOT remove high-price vehicles ---
# Luxury and exotic cars (e.g. $200,000+) are valid real-world listings.
# Removing them would make the model blind to the premium segment and
# introduce systematic underestimation bias for expensive vehicles.

n_before = len(df)
df = df[df['Price'] >= 500].copy()
print(f'Removed {n_before - len(df)} rows with Price < $500')
print(f'Dataset size after filtering: {len(df)}')

# Visualise effect
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(np.log1p(df['Price']), bins=80, color=sns.color_palette('Set2')[2], edgecolor='none')
ax.set_title('log(1 + Price) Distribution — After Removing Erroneous Low Prices')
ax.set_xlabel('log(1 + Price)')
ax.set_ylabel('Count')
sns.despine()
plt.tight_layout()
plt.savefig('fig03_price_distribution_clean.png', bbox_inches='tight')
plt.show()

## 5. Train / Test Split

**The split is performed here — before feature engineering.** This is critical: if we engineered features on the full dataset first and then split, statistics from the test set would "leak" into the training transformers, making our evaluation metrics unrealistically optimistic.

Because the price distribution is heavily right-skewed, a simple random split may place most high-value vehicles in one set. We prevent this by **stratifying on price quantile bins**, ensuring both sets reflect the same price composition.

In [ ]:
# Assign each row to a price decile (10 equal-frequency bins)
df['_price_bin'] = pd.qcut(df['Price'], q=10, labels=False, duplicates='drop')

X = df.drop(columns=['Price', '_price_bin'])
y = df['Price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=df['_price_bin']
)

print(f'Training samples : {len(X_train):,}  ({len(X_train)/len(X)*100:.0f}%)')
print(f'Test samples     : {len(X_test):,}  ({len(X_test)/len(X)*100:.0f}%)')

In [ ]:
# Verify the stratification worked — both sets should show similar distributions
fig, ax = plt.subplots(figsize=(9, 4))
bins = np.linspace(np.log1p(y.min()), np.log1p(y.max()), 45)
ax.hist(np.log1p(y_train), bins=bins, alpha=0.65, label=f'Train (n={len(y_train):,})',
        color=sns.color_palette('Set2')[0])
ax.hist(np.log1p(y_test), bins=bins, alpha=0.65, label=f'Test  (n={len(y_test):,})',
        color=sns.color_palette('Set2')[3])
ax.set_xlabel('log(1 + Price)')
ax.set_ylabel('Count')
ax.set_title('Target Distribution: Train vs Test (Stratified 80/20 Split)')
ax.legend()
sns.despine()
plt.tight_layout()
plt.savefig('fig04_train_test_split.png', bbox_inches='tight')
plt.show()

print('\nTrain price stats:')
print(y_train.describe().round(0))
print('\nTest price stats:')
print(y_test.describe().round(0))

## 6. Feature Engineering

We create **4 hand-crafted features** using domain knowledge about used car valuation. All are derived purely from existing columns — no external data is introduced.

| Feature | Formula | Reasoning |
|---|---|---|
| `Car_Age` | `2024 − Prod. year` | Directly captures depreciation timeline |
| `Km_Per_Year` | `Mileage ÷ (Car_Age + 1)` | Normalises wear intensity; low km/year on an old car is a positive signal |
| `Has_Turbo` | 1 if engine string contains "Turbo" | Turbocharged engines are a common premium indicator |
| `Airbag_Tier` | Binned airbag count (0, low, mid, high) | Safety tier rather than raw count reduces the effect of outlier configurations |

In [ ]:
# Store the raw engine volume string before it was cleaned — needed for Turbo flag
raw_engine_str = df_raw['Engine volume'].astype(str)

def add_features(X_in):
    X_out = X_in.copy()

    # Feature 1: Car Age
    X_out['Car_Age'] = (2024 - X_out['Prod. year'].astype(int)).clip(lower=0)

    # Feature 2: Kilometres Per Year of ownership
    X_out['Km_Per_Year'] = X_out['Mileage'] / (X_out['Car_Age'] + 1)

    # Feature 3: Turbo flag — extracted from original raw string
    X_out['Has_Turbo'] = (
        raw_engine_str.loc[X_in.index]
        .str.lower()
        .str.contains('turbo')
        .astype(int)
    )

    # Feature 4: Airbag Tier — binned from raw airbag count
    # Bins: 0 airbags → tier 0, 1-4 → tier 1, 5-8 → tier 2, 9+ → tier 3
    X_out['Airbag_Tier'] = pd.cut(
        X_out['Airbags'],
        bins=[-1, 0, 4, 8, 100],
        labels=[0, 1, 2, 3]
    ).astype(int)

    return X_out

X_train = add_features(X_train)
X_test  = add_features(X_test)

new_feats = ['Car_Age', 'Km_Per_Year', 'Has_Turbo', 'Airbag_Tier']
print('New features added:', new_feats)
X_train[new_feats].describe().round(2)

## 7. Analysis of New Features

We analyse each new feature with the same methods used in M1: distribution plots and relationship to the target variable.

In [ ]:
# --- Distributions of continuous new features ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
palette = sns.color_palette('Set2')

sns.histplot(X_train['Car_Age'], kde=True, ax=axes[0], color=palette[0], bins=30)
axes[0].set_title('Distribution of Car Age (years)')
axes[0].set_xlabel('Car Age')
axes[0].set_ylabel('Count')

# Clip extreme Km_Per_Year for readability (a few very old low-mileage cars create long tail)
kpy_clipped = X_train['Km_Per_Year'].clip(upper=X_train['Km_Per_Year'].quantile(0.98))
sns.histplot(kpy_clipped, kde=True, ax=axes[1], color=palette[1], bins=40)
axes[1].set_title('Distribution of Km Per Year (98th pct clipped)')
axes[1].set_xlabel('Km / Year')
axes[1].set_ylabel('Count')

plt.suptitle('New Continuous Features — Training Set Distributions', fontweight='bold')
sns.despine()
plt.tight_layout()
plt.savefig('fig05_new_feature_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- New features vs Price: violin plots ---
# We use violin plots because they show both the distribution shape and the
# median simultaneously — more informative than a plain scatter for categorical splits.

train_plot = X_train[['Has_Turbo', 'Airbag_Tier']].copy()
train_plot['Price'] = y_train.values
train_plot['Has_Turbo'] = train_plot['Has_Turbo'].map({0: 'Non-Turbo', 1: 'Turbo'})

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.violinplot(data=train_plot, x='Has_Turbo', y='Price', palette='Set2',
               ax=axes[0], inner='quartile', cut=0)
axes[0].set_title('Price by Engine Type (Turbo vs Non-Turbo)')
axes[0].set_xlabel('Engine Type')
axes[0].set_ylabel('Price ($)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

sns.violinplot(data=train_plot, x='Airbag_Tier', y='Price', palette='Set2',
               ax=axes[1], inner='quartile', cut=0)
axes[1].set_title('Price by Airbag Safety Tier (0=none, 3=high)')
axes[1].set_xlabel('Airbag Tier')
axes[1].set_ylabel('Price ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.suptitle('New Categorical Features vs Price — Training Set', fontweight='bold')
sns.despine()
plt.tight_layout()
plt.savefig('fig06_new_features_vs_price_violin.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- KDE plot: Car Age vs Price ---
# We split into age terciles to see how price density shifts across age groups
train_plot2 = pd.DataFrame({
    'Car_Age': X_train['Car_Age'],
    'Price': y_train.values
})
terciles = pd.qcut(train_plot2['Car_Age'], q=3, labels=['Young (0-6yr)', 'Mid (7-13yr)', 'Old (14+yr)'])
train_plot2['Age Group'] = terciles

fig, ax = plt.subplots(figsize=(10, 4))
for group, color in zip(['Young (0-6yr)', 'Mid (7-13yr)', 'Old (14+yr)'],
                         sns.color_palette('Set2')):
    subset = train_plot2[train_plot2['Age Group'] == group]['Price']
    sns.kdeplot(np.log1p(subset), ax=ax, label=group, fill=True, alpha=0.3, color=color)
ax.set_xlabel('log(1 + Price)')
ax.set_ylabel('Density')
ax.set_title('Price Density by Car Age Group (KDE on log-price)')
ax.legend(title='Age Group')
sns.despine()
plt.tight_layout()
plt.savefig('fig07_car_age_kde.png', bbox_inches='tight')
plt.show()

**Observations:**
- Turbo-engined cars have a noticeably higher median price and a wider upper tail, confirming the feature is a useful price signal.
- Higher airbag tiers correspond to higher median prices, consistent with the M1 finding that safety specs drive value.
- Younger cars clearly dominate the higher price range; the KDE peaks shift left as age increases, confirming strong depreciation.

## 8. Encoding Categorical Features

Machine learning models require numeric input. We handle categorical columns as follows:

- **High-cardinality** (Manufacturer, Model, Color — many unique values): **Ordinal Encoding**. One-hot encoding would add hundreds of sparse columns.
- **Low-cardinality** (Fuel type, Gear box type, Drive wheels, Leather interior — few unique values): **One-Hot Encoding**. Creates interpretable binary features with no imposed ordinal relationship.

In [ ]:
cat_cols = X_train.select_dtypes('object').columns.tolist()
cardinality = {c: X_train[c].nunique() for c in cat_cols}
print('Cardinality of categorical columns:')
for col, n in sorted(cardinality.items(), key=lambda x: -x[1]):
    enc = 'Ordinal' if n > 8 else 'One-Hot'
    print(f'  {col:25s} {n:4d} unique  →  {enc}')

high_card = [c for c, n in cardinality.items() if n > 8]
low_card  = [c for c, n in cardinality.items() if n <= 8]

In [ ]:
# Ordinal encoding for high-cardinality features
# Unknown test-set categories are assigned -1
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[high_card] = oe.fit_transform(X_train[high_card].astype(str))
X_test[high_card]  = oe.transform(X_test[high_card].astype(str))

# One-hot encoding for low-cardinality features
X_train = pd.get_dummies(X_train, columns=low_card, drop_first=True)
X_test  = pd.get_dummies(X_test,  columns=low_card, drop_first=True)

# Align columns — test set may be missing some dummy columns if a rare
# category only appears in the training set
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

print(f'Feature count after encoding: {X_train.shape[1]}')
X_train.head(3)

## 9. Scaling

We apply **MinMaxScaler**, which maps all numeric features to the [0, 1] range. This is a good fit for this dataset because:
- It preserves the shape of each feature's distribution
- Tree-based models (like the GBM we plan to use in M3) are not affected by scale, but distance-based and linear models benefit greatly
- We want all features on a common scale for fair mutual comparison

**Crucially, the scaler is fitted only on the training set** and then applied to the test set — this prevents any test-set information from influencing the transformation.

In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()

scaler = MinMaxScaler()
X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()

X_train_scaled[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test_scaled[numeric_features]  = scaler.transform(X_test[numeric_features])

# Visualise before vs after for 3 key features
demo_feats = ['Mileage', 'Car_Age', 'Levy']
fig, axes = plt.subplots(2, 3, figsize=(13, 7))

for i, feat in enumerate(demo_feats):
    sns.histplot(X_train[feat], bins=35, ax=axes[0, i],
                 color=sns.color_palette('Set2')[i], kde=False)
    axes[0, i].set_title(f'{feat} — Original Scale')
    axes[0, i].set_xlabel(feat)

    sns.histplot(X_train_scaled[feat], bins=35, ax=axes[1, i],
                 color=sns.color_palette('Set2')[i+3], kde=False)
    axes[1, i].set_title(f'{feat} — After MinMaxScaler [0,1]')
    axes[1, i].set_xlabel(f'{feat} (scaled)')

plt.suptitle('MinMaxScaler Effect on Key Features (shape preserved)', fontweight='bold')
sns.despine()
plt.tight_layout()
plt.savefig('fig08_scaling_effect.png', bbox_inches='tight')
plt.show()

## 10. Feature Selection via Permutation Importance

We fit a lightweight Gradient Boosting model on the training set and compute **Permutation Importance** — a model-agnostic technique that measures how much the model's error increases when a feature's values are randomly shuffled. A large increase means the model relied on that feature; near-zero increase means the feature is not useful.

This is different from built-in tree importances because it evaluates features on **held-out validation data**, making it a more honest estimate of real predictive value.

In [ ]:
# Fit a fast GBM for importance estimation only (not the final model)
gbm = GradientBoostingRegressor(
    n_estimators=80,
    max_depth=4,
    learning_rate=0.1,
    random_state=RANDOM_SEED
)
gbm.fit(X_train_scaled.fillna(0), y_train)

# Permutation importance on a validation slice of the training set
perm = permutation_importance(
    gbm,
    X_train_scaled.fillna(0),
    y_train,
    n_repeats=10,
    random_state=RANDOM_SEED,
    n_jobs=-1
)

perm_df = pd.DataFrame({
    'importance_mean': perm.importances_mean,
    'importance_std':  perm.importances_std
}, index=X_train_scaled.columns).sort_values('importance_mean', ascending=False)

print('Top 15 features by permutation importance:')
print(perm_df.head(15).round(4))

In [ ]:
top15 = perm_df.head(15).sort_values('importance_mean')

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(
    top15.index,
    top15['importance_mean'],
    xerr=top15['importance_std'],
    color=sns.color_palette('Set2')[0],
    ecolor='grey',
    capsize=3
)
ax.set_xlabel('Mean Increase in MSE when Feature is Shuffled')
ax.set_title('Top 15 Features — Permutation Importance (with std error bars)')
sns.despine()
plt.tight_layout()
plt.savefig('fig09_permutation_importance.png', bbox_inches='tight')
plt.show()

In [ ]:
# Keep features with mean importance > 0 (drop those that add noise)
selected = perm_df[perm_df['importance_mean'] > 0].index.tolist()
print(f'Features selected: {len(selected)} out of {X_train_scaled.shape[1]}')

X_train_final = X_train_scaled[selected].copy()
X_test_final  = X_test_scaled[selected].copy()

print(f'\nFinal training set : {X_train_final.shape}')
print(f'Final test set     : {X_test_final.shape}')

## 11. Correlation Matrix — Final Feature Set

In [ ]:
# Show top 12 features for readability
top12 = perm_df.head(12).index.tolist()
corr_data = X_train_final[top12].copy()
corr_data['Price'] = y_train.values

fig, ax = plt.subplots(figsize=(11, 9))
corr = corr_data.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0,
    linewidths=0.4, ax=ax,
    cbar_kws={'label': 'Pearson r', 'shrink': 0.8}
)
ax.set_title('Correlation Matrix — Top 12 Features + Price', fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('fig10_correlation_matrix_final.png', bbox_inches='tight')
plt.show()

## 12. Export Prepared Data

In [ ]:
X_train_final.to_csv('X_train_prepared.csv', index=False)
X_test_final.to_csv('X_test_prepared.csv',   index=False)
y_train.to_csv('y_train_prepared.csv',        index=False)
y_test.to_csv('y_test_prepared.csv',          index=False)

print('Files saved:')
print(f'  X_train_prepared.csv  →  {X_train_final.shape}')
print(f'  X_test_prepared.csv   →  {X_test_final.shape}')
print(f'  y_train_prepared.csv  →  {y_train.shape}')
print(f'  y_test_prepared.csv   →  {y_test.shape}')

## Summary

| Step | Method | Key Decision |
|------|--------|-------------|
| Type cleaning | Strip units, coerce strings → numeric | Required for `Mileage`, `Engine volume`, `Levy`, `Doors` |
| Missing values | Median imputation (Levy); row drop (categoricals) | Median preserves skewed distributions better than mean |
| Outlier filtering | Remove Price < $500 only | High-priced luxury cars kept — they are valid data points |
| Train/test split | 80/20, stratified by price decile | Prevents class imbalance in rare high-price segment |
| New features | `Car_Age`, `Km_Per_Year`, `Has_Turbo`, `Airbag_Tier` | Domain-driven; all show meaningful relationship with price |
| Encoding | Ordinal (high-card) + OHE (low-card) | Avoids explosion of sparse columns |
| Scaling | MinMaxScaler (fit on train only) | No test leakage; shape-preserving normalisation |
| Feature selection | Permutation Importance (GBM baseline) | Model-agnostic; evaluated on training data |

The processed dataset is ready for model training in **Milestone 3**.